In [1]:
import shutil
import tarfile
from pathlib import Path

archive = Path("../data/processed/pylate_index.tar.gz")
index_dir = Path("../data/processed/pylate_index")

if not index_dir.exists():
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(path="../data/processed")

In [2]:
# Initialize the colBERT retriever

from pylate import retrieve
from pylate import indexes, models
index = indexes.Voyager(
    index_folder="../data/processed/pylate_index",
    index_name="esci_data_index",
)

retriever = retrieve.ColBERT(index=index)

In [3]:
import torch

# Encode the query
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

model = models.ColBERT(
    model_name_or_path="lightonai/colbertv2.0",
    device = device
)
query_text = "energy efficient fan"

query_embeddings = model.encode(
    [query_text],
    batch_size=1,
    is_query=True,
    show_progress_bar=False
)

Using device: mps


In [4]:
# Retrieve and map the results

results = retriever.retrieve(
    queries_embeddings=query_embeddings,
    k=3
)
print(results)

Retrieving documents (bs=50): 100%|██████████| 1/1 [00:00<00:00,  4.39it/s]

[[{'id': 6, 'score': 23.474761962890625}, {'id': 461, 'score': 22.531002044677734}, {'id': 1, 'score': 22.443485260009766}]]


In [5]:
# Display the results from SQLite database
import sqlite3

conn = sqlite3.connect("../data/processed/products_dataset.db")
cursor = conn.cursor()

print(f"Top results for query: {query_text}")
for hit in results[0]:
    product_id = hit["id"]
    score = hit["score"]

    cursor.execute("SELECT product_title FROM products WHERE pid = ?", (product_id,))
    title = cursor.fetchone()[0]
    print(f"[{score:.2f}] : {product_id} - {title}")

conn.close()

Top results for query: energy efficient fan
[23.47] : 6 - Panasonic FV-08VRE2 Ventilation Fan with Recessed LED (Renewed)
[22.53] : 461 - 52 Inch Modern Ceiling Fan with Light Ceiling Fan Chandelier with Remote Control 3 Speed Reverse Function without Light Source E26 Suitable for Living Room, Bedroom, Dining Room
[22.44] : 1 - Aero Pure AP80RVLW Super Quiet 80 CFM Recessed Fan/Light Bathroom Ventilation Fan with White Trim Ring


In [6]:
# Clean up extracted index to save disk space
if index_dir.exists():
    shutil.rmtree(index_dir)
    print(f"Deleted extracted index: {index_dir}")

Deleted extracted index: ../data/processed/pylate_index
